이 노트북은 예제 렌더링 HDF5 파일들을 탐색하여 내부의 RGB, depth, normal 이미지를 PNG로 추출하고,
이미지 파일 경로와 카메라 변환 행렬(transform_matrix)을 포함하는 NeRF 스타일의 `transforms_train.json`을 생성합니다.

사용법 (1~2번 셀을 순서대로 실행):
1) 파라미터(렌더링 폴더, 출력 폴더 등)를 수정하여 셀을 실행합니다.
2) 변환 행렬과 이미지는 출력 폴더 아래에 저장됩니다. RGB는 `train/`, depth는 `depth/`, normal은 `normal/` 폴더에 저장됩니다.

옵션:
- depth_scale: 깊이를 정수 PNG로 저장할 때 곱할 스케일(기본 1000, 미터->밀리미터).
- flip_x: 이미지 저장 시 좌우(또는 상하) 반전을 수행해야 하는 경우 True로 설정(렌더 세팅에 따라 필요할 수 있음).
- cam_K_path: 카메라 내부행렬(cam_K.npy) 경로를 지정하면 `camera_angle_x`를 transforms json에 계산하여 추가합니다.
</VSCode.Cell>

<VSCode.Cell language="python">
# HDF5 렌더 파일을 PNG로 분리하고 transforms json 생성m

In [2]:
import os
from pathlib import Path
import json
import h5py
import numpy as np
from PIL import Image
from math import atan


def _save_rgb(arr, path, flip_x=False):
    if flip_x:
        arr = arr[:, ::-1]
    if arr.dtype == np.float32 or arr.dtype == np.float64:
        if arr.max() <= 1.01:
            arr = (arr * 255.0).astype(np.uint8)
        else:
            arr = np.clip(arr, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr.astype(np.uint8))
    img.save(str(path))


def _visualize_depth_to_rgb(depth_arr, cmap_name='viridis', clip_percentiles=(1, 99), assume_mm_threshold=50.0):
    """
    Create 3-channel RGB visualization for depth:
    - Saves as uint8 RGB image suitable for quick inspection.
    - Uses percentile clipping to avoid outlier impact.
    - Auto-converts mm->m when masked max exceeds assume_mm_threshold.
    - Invalid/non-positive pixels are set to black.
    """
    try:
        import matplotlib.cm as cm
        cmap = cm.get_cmap(cmap_name)
        has_cmap = True
    except Exception:
        cmap = None
        has_cmap = False

    depth = np.array(depth_arr, dtype=np.float32)
    # mask valid depths > 0
    mask = np.isfinite(depth) & (depth > 0)
    h, w = depth.shape[:2]
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    if not mask.any():
        return rgb

    vals = depth[mask]
    # auto detect mm values
    if float(vals.max()) > assume_mm_threshold:
        depth = depth / 1000.0
        vals = depth[mask]

    try:
        vmin, vmax = np.percentile(vals, [clip_percentiles[0], clip_percentiles[1]])
    except Exception:
        vmin, vmax = float(vals.min()), float(vals.max())

    if vmax <= vmin:
        vmin, vmax = float(vals.min()), float(vals.max())
        if vmax <= vmin:
            norm = np.zeros_like(depth)
        else:
            norm = (depth - vmin) / (vmax - vmin)
    else:
        norm = (depth - vmin) / (vmax - vmin)

    norm = np.clip(norm, 0.0, 1.0)

    if has_cmap and cmap is not None:
        mapped = cmap(norm)[:, :, :3]
        rgb = (mapped * 255.0).astype(np.uint8)
        rgb[~mask] = 0
    else:
        gray = (norm * 255.0).astype(np.uint8)
        rgb = np.stack([gray, gray, gray], axis=-1)
        rgb[~mask] = 0

    return rgb


def _save_depth(arr, path_base, depth_scale=1000.0, flip_x=False, clip_percentiles=(1,99), assume_mm_threshold=50.0):
    """
    Save depth in three forms:
    - Raw GT numpy (.npy) saved unchanged.
    - Visualized RGB PNG (.png) produced by _visualize_depth_to_rgb.
    - 16-bit PNG scaled by depth_scale for downstream tools (_uint16.png).
    """
    if flip_x:
        arr = arr[:, ::-1]
    arr = np.array(arr, dtype=np.float32)

    # Save raw GT depth as .npy
    npy_path = Path(str(path_base) + '.npy')
    np.save(str(npy_path), arr)

    # Visualized 3-channel PNG
    rgb_viz = _visualize_depth_to_rgb(arr, cmap_name='viridis', clip_percentiles=clip_percentiles, assume_mm_threshold=assume_mm_threshold)
    viz_path = Path(str(path_base) + '.png')
    Image.fromarray(rgb_viz).save(str(viz_path))

    # Also save a 16-bit depth image scaled by depth_scale (useful for some tools)
    depth_int = (np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0) * depth_scale).clip(0, 65535).astype(np.uint16)
    uint16_path = Path(str(path_base) + '_uint16.png')
    Image.fromarray(depth_int).save(str(uint16_path))


def _save_normal(arr, path, flip_x=False):
    if flip_x:
        arr = arr[:, ::-1]
    if arr.dtype == np.float32 or arr.dtype == np.float64:
        if arr.min() < -0.1:
            arr = (arr * 0.5 + 0.5) * 255.0
        else:
            arr = arr * 255.0
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr)
    img.save(str(path))


def process_render_folder(render_dir, out_dir, depth_scale=1000.0, flip_x=False, cam_K_path=None, convert_blender_to_opengl=False, depth_clip_percentiles=(1,99), assume_mm_threshold=50.0):
    render_dir = Path(render_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train_dir = out_dir.joinpath('train')
    depth_dir = out_dir.joinpath('depth')
    normal_dir = out_dir.joinpath('normal')
    train_dir.mkdir(exist_ok=True)
    depth_dir.mkdir(exist_ok=True)
    normal_dir.mkdir(exist_ok=True)

    # accept both .h5 and .hdf5
    h5_paths = sorted(list(render_dir.glob('*.h5')) + list(render_dir.glob('*.hdf5')))
    if len(h5_paths) == 0:
        print('No .h5/.hdf5 files found in', render_dir)
        return

    frames = []
    cam_angle_x = None

    # If cam_K provided, compute camera_angle_x
    if cam_K_path is not None and Path(cam_K_path).exists():
        try:
            K = np.load(cam_K_path)
            fx = float(K[0, 0])
        except Exception:
            fx = None
    else:
        fx = None

    idx = 0
    for h5_path in h5_paths:
        with h5py.File(h5_path, 'r') as f:
            color = np.array(f['colors']) if 'colors' in f else None
            depth = np.array(f['depth']) if 'depth' in f else None
            normals = np.array(f['normals']) if 'normals' in f else None
            cam_T = np.array(f['cam_Ts']) if 'cam_Ts' in f else None

        if color is not None:
            if color.ndim == 3 and color.shape[0] in (3,4) and color.shape[2] not in (3,4):
                color = np.transpose(color, (1,2,0))

        if normals is not None and normals.ndim == 3 and normals.shape[0] in (3,):
            normals = np.transpose(normals, (1,2,0))

        if idx == 0 and fx is not None and color is not None:
            width = color.shape[1]
            cam_angle_x = 2.0 * atan(width / (2.0 * fx))

        rgb_name = train_dir.joinpath(f"{idx:04d}.png")
        depth_base = depth_dir.joinpath(f"{idx:04d}")
        normal_name = normal_dir.joinpath(f"{idx:04d}.png")

        if color is not None:
            _save_rgb(color, rgb_name, flip_x=flip_x)
        if depth is not None:
            _save_depth(depth, depth_base, depth_scale=depth_scale, flip_x=flip_x, clip_percentiles=depth_clip_percentiles, assume_mm_threshold=assume_mm_threshold)
        if normals is not None:
            _save_normal(normals, normal_name, flip_x=flip_x)

        if cam_T is not None:
            if cam_T.shape == (4,4):
                mat = cam_T.tolist()
            else:
                try:
                    mat = np.array(cam_T).reshape(4,4).tolist()
                except Exception:
                    mat = None
        else:
            mat = None

        frames.append({
            'file_path': str(Path('train').joinpath(f"{idx:04d}.png")),
            'transform_matrix': mat if mat is not None else []
        })

        idx += 1

    out_json = {
        'camera_angle_x': float(cam_angle_x) if cam_angle_x is not None else None,
        'frames': frames
    }

    json_path = out_dir.joinpath('transforms_train.json')
    with open(json_path, 'w') as jf:
        json.dump(out_json, jf, indent=4)

    print('Saved', idx, 'frames to', out_dir)
    print('transforms json saved to', json_path)
    
# Example: 사용 예시 (경로를 실제 값으로 바꿔서 실행하세요)
example_render_dir = '/home/user/ssd/Codes/BlenderProc-3DFront/examples/datasets/front_3d_with_improved_mat/renderings_single_test_light_camera_centered'
example_out_dir = './dataset_formatted_improved_cam_centered/'

for scene in os.listdir(example_render_dir):
    if os.path.isdir(os.path.join(example_render_dir, scene)):
        scene_path = os.path.join(example_render_dir, scene)
        out_scene_path = os.path.join(example_out_dir, scene)
        
        print(f'Processing scene: {scene}')
        process_render_folder(scene_path, out_scene_path, depth_scale=1000.0, flip_x=False, cam_K_path=f'{scene_path}/cam_K.npy')
        